# Sampling Responses from Claude Beyond the Max Tokens Limit

Every Messages API request sets `max_tokens`, a ceiling on how much Claude may generate. When a response reaches that ceiling the API doesn't fail: it returns what Claude wrote so far and sets `stop_reason` to `"max_tokens"`. If you don't check for it, you ship half an answer.

This notebook shows how to detect a cut-off response and get the rest of it:

1. Give Claude enough room in the first place, and stream long responses.
2. When a response is still cut off, ask Claude to continue in a follow-up turn and stitch the pieces together.

> **What changed:** earlier versions of this notebook put the partial response back as the *final* assistant message (a "prefill") so Claude would continue it. Claude models from the 4.6 generation onward [reject prefill](https://platform.claude.com/docs/en/build-with-claude/working-with-messages#putting-words-in-claudes-mouth) with a `400` ("This model does not support assistant message prefill"). The continuation pattern below ends every request with a user message, so it works on all current models.

In [1]:
%%capture
%pip install anthropic

In [2]:
import re

import anthropic

# Reads your API key from the ANTHROPIC_API_KEY environment variable.
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

## 1. A response that runs out of room

We ask for something long and deliberately cap `max_tokens` at 600 so the cut-off is quick and cheap to reproduce. In a real application the ceiling is more likely to be the model's output limit or a budget you chose.

In [3]:
PROMPT = """Write three short stories, each about a different animal and each roughly 300 words long.
Put them in <story_1>, <story_2>, and <story_3> tags."""

first = client.messages.create(
    model=MODEL,
    max_tokens=600,
    messages=[{"role": "user", "content": PROMPT}],
)
print("stop_reason:", first.stop_reason)
print("output tokens:", first.usage.output_tokens)

stop_reason: max_tokens
output tokens: 600


`stop_reason` is `"max_tokens"` and the output used the whole budget. The text stops mid-thought:

In [4]:
first_text = "".join(block.text for block in first.content if block.type == "text")
print("..." + first_text[-400:])

...fined in him that might, in a human, have been called memory.

Tonight his sonar found a large squid moving in slow spirals forty meters below. He adjusted course without urgency and drove downward. The squid was nearly two meters long, substantial enough to matter. The chase lasted less than a minute.

He had been diving for forty-five minutes and still had time before he needed air, but he began


## 2. First, give Claude enough room

`max_tokens` is a ceiling, not a target: you're billed for the tokens Claude actually generates, so a generous value costs nothing when the answer is short. Current models can produce far more output per request than the 4,096 tokens this notebook was originally written around; each model's limit is listed in the [models overview](https://platform.claude.com/docs/en/about-claude/models/overview).

For long outputs, [stream the response](https://platform.claude.com/docs/en/build-with-claude/streaming). You can show text as it arrives, and the SDKs require streaming for requests that could run for many minutes. With enough room, the same prompt finishes on its own:

In [5]:
with client.messages.stream(
    model=MODEL,
    max_tokens=8000,
    messages=[{"role": "user", "content": PROMPT}],
) as stream:
    received_chars = 0
    for text in stream.text_stream:
        received_chars += len(text)  # a real app would render `text` as it arrives
    complete = stream.get_final_message()

print("streamed characters:", received_chars)
print("stop_reason:", complete.stop_reason)
print("output tokens:", complete.usage.output_tokens)

streamed characters: 5144
stop_reason: end_turn
output tokens: 1207


## 3. Continue a response that was cut off

Sometimes raising the ceiling isn't an option: the answer is longer than the model's output limit, or you cap each request on purpose. In that case, keep the partial reply in the conversation as the assistant turn it was, then add a **user** turn asking Claude to pick up where it stopped. Repeat until `stop_reason` is no longer `"max_tokens"` and join the pieces. This is the pattern the [stop reasons guide](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#ensuring-complete-responses) recommends.

One detail makes the joins clean. A `max_tokens` cut can land in the middle of a word, and when Claude resumes it doesn't reproduce the space or line break that belonged at the cut. So before asking for more, the helper trims the partial reply back to its last whitespace: Claude then always resumes at the start of a word, and the whitespace we kept is the separator for the join.

In [6]:
CONTINUE_INSTRUCTION = (
    "Your previous reply was cut off because it reached the output token limit. "
    "Continue with the next word from exactly where it stops. Do not repeat anything you "
    "already wrote and do not add any introduction; your reply will be appended directly "
    "to the previous text."
)


def complete_response(prompt: str, max_tokens: int, max_rounds: int = 6) -> str:
    """Sample a full response in pieces of at most `max_tokens` each."""
    text = ""
    messages = [{"role": "user", "content": prompt}]
    for round_number in range(1, max_rounds + 1):
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, messages=messages)
        piece = "".join(block.text for block in response.content if block.type == "text")
        print(
            f"round {round_number}: stop_reason={response.stop_reason}, "
            f"output_tokens={response.usage.output_tokens}"
        )
        if text:
            piece = piece.lstrip()  # `text` already ends with the separator we kept
        if response.stop_reason != "max_tokens":
            return text + piece
        # Drop the (possibly partial) last word but keep the whitespace before it.
        trailing = re.search(r"(\s+)\S*$", piece)
        if trailing:
            piece = piece[: trailing.end(1)]
        text += piece
        # Claude's partial reply goes back as an assistant turn; the request still ends
        # with a user turn, so this is ordinary conversation history, not a prefill.
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": text.rstrip()},
            {"role": "user", "content": CONTINUE_INSTRUCTION},
        ]
    return text


full_text = complete_response(PROMPT, max_tokens=600)

round 1: stop_reason=max_tokens, output_tokens=600


round 2: stop_reason=max_tokens, output_tokens=600


round 3: stop_reason=end_turn, output_tokens=410


Claude picks up mid-sentence and the stories come out whole:

In [7]:
print(full_text)

<story_1>
**The Fox and the Frost**

The red fox moved through the pine forest like a flame blown sideways by the wind. Her name, if foxes had names, would have been something quick and clever. She had survived three winters in these woods, which was no small thing.

This particular morning, the ground was sealed beneath a sheet of ice that had formed overnight when the rain turned cold mid-fall. She could hear the mice beneath it — their tiny heartbeats and scratching claws — but her paws slid uselessly across the surface every time she tried to pounce.

She sat back on her haunches and tilted her head.

Other foxes might have given up, might have trotted toward the farmhouse a mile east where chickens made easy, if dangerous, targets. But she had learned patience the hard way, after losing two toes to a trap near that farmhouse two winters ago.

She walked along a fallen log instead, using it as a raised platform to scan the clearing. Near the boulder at the forest's edge, the ice ha

## Things to keep in mind

- **Check the result when format matters.** The word-boundary trim keeps prose seams clean, but Claude is still reconstructing where it was from context. For output that must be exact (code, JSON, tables), prefer a larger `max_tokens` with streaming over stitching, or validate the joined result.
- **Cost.** Each continuation round sends the prompt and every earlier piece back as input tokens, so you pay for the earlier output again as input (output tokens are only billed once). For long prompts, [prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) keeps the repeated input cheap.
- **Cap the rounds.** The `max_rounds` guard stops a runaway loop if the task is open-ended. If you hit it regularly, the request needs a bigger budget, not more rounds.